In [167]:


import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text

from nhs_waiting_lists.constants import proj_db_path
from nhs_waiting_lists.utils.proj_paths import find_project_root

project_root = find_project_root()

DB_PATH = project_root / proj_db_path / "nhs_rttwtd.db"
DATA_DIR = "./data"

conn = create_engine(f"sqlite:///{DB_PATH}")

In [168]:

from nhs_waiting_lists.utils.utils2 import last_full_quarter

print(last_full_quarter("2025-7"))

(2025, 2)


In [169]:


query = text("""
             SELECT *
             from v_consolidated; \
             """).bindparams(
    # bindparam('provider_codes', expanding=True),
    # bindparam('treatment_codes', expanding=True)
)

df = pd.read_sql(
    query, conn, params={  # type: ignore[arg-type]
        # 'provider_codes': PROVIDER_CODES,
    }
)

providers = pd.read_sql("""
                        SELECT DISTINCT provider_code AS provider, provider_name
                        FROM providers
                        """, conn, params={  # type: ignore[arg-type]
    # 'provider_codes': PROVIDER_CODES,
})

# df.query("provider == 'RAJ' and treatment == 'C_999' and period == '2025-08'")
df

,period,provider_name,subtype,Quarter,provider,treatment,incomplete,admitted,nonadmitted,new_periods,incomplete_prev,wait_pct_lt_18
0,2021-05,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,2,R0B,C_999,38116,2469,10814,17215,NaN,0.875669
1,2021-06,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,2,R0B,C_999,40317,2772,11371,18622,38116.0,0.881812
2,2021-07,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,3,R0B,C_999,43170,2420,10116,17791,40317.0,0.869099
3,2021-08,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,3,R0B,C_999,43946,2356,8898,15761,43170.0,0.856528
4,2021-09,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,3,R0B,C_999,45349,2790,10123,18007,43946.0,0.844848
...,...,...,...,...,...,...,...,...,...,...,...,...
1186,2025-04,East Lancashire Hospitals NHS Trust,Acute - Large,2,RXR,C_999,60237,1755,7215,11017,60809.0,0.564620
1187,2025-05,East Lancashire Hospitals NHS Trust,Acute - Large,2,RXR,C_999,57660,2012,7897,11771,60237.0,0.584287
1188,2025-06,East Lancashire Hospitals NHS Trust,Acute - Large,2,RXR,C_999,58192,2172,8484,13750,57660.0,0.606853
1189,2025-07,East Lancashire Hospitals NHS Trust,Acute - Large,3,RXR,C_999,57371,2368,8695,14741,58192.0,0.611685


In [170]:
import pandas as pd

dt_recent = pd.read_sql("""
                        SELECT MAX(period) as period
                        FROM consolidated
                        """, conn, params={  # type: ignore[arg-type]
})

print(last_full_quarter(dt_recent["period"].iloc[0]))

(2025, 2)


In [171]:
from nhs_waiting_lists.utils.utils2 import last_n_full_quarters

full_quarters = last_n_full_quarters(dt_recent["period"].iloc[0])
full_quarters

[(2024, 4), (2025, 1), (2025, 2)]

In [172]:
# unique years in the quarters
years = list(set([x[0] for x in full_quarters]))
years

[2024, 2025]

In [173]:


# extract year + quarter
df["year"] = df["period"].str[:4].astype(int)
df["month"] = df["period"].str[5:7].astype(int)
df["quarter"] = df['year'].astype(str) + '-q' + ((df['month'] - 1) // 3 + 1).astype(str)

# df["quarter"] = f"{df['year']}-{((df['month'] - 1) // 3) + 1}"

df

,period,provider_name,subtype,Quarter,provider,treatment,incomplete,admitted,nonadmitted,new_periods,incomplete_prev,wait_pct_lt_18,year,month,quarter
0,2021-05,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,2,R0B,C_999,38116,2469,10814,17215,NaN,0.875669,2021,5,2021-q2
1,2021-06,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,2,R0B,C_999,40317,2772,11371,18622,38116.0,0.881812,2021,6,2021-q2
2,2021-07,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,3,R0B,C_999,43170,2420,10116,17791,40317.0,0.869099,2021,7,2021-q3
3,2021-08,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,3,R0B,C_999,43946,2356,8898,15761,43170.0,0.856528,2021,8,2021-q3
4,2021-09,South Tyneside and Sunderland NHS Foundation T...,Acute - Large,3,R0B,C_999,45349,2790,10123,18007,43946.0,0.844848,2021,9,2021-q3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1186,2025-04,East Lancashire Hospitals NHS Trust,Acute - Large,2,RXR,C_999,60237,1755,7215,11017,60809.0,0.564620,2025,4,2025-q2
1187,2025-05,East Lancashire Hospitals NHS Trust,Acute - Large,2,RXR,C_999,57660,2012,7897,11771,60237.0,0.584287,2025,5,2025-q2
1188,2025-06,East Lancashire Hospitals NHS Trust,Acute - Large,2,RXR,C_999,58192,2172,8484,13750,57660.0,0.606853,2025,6,2025-q2
1189,2025-07,East Lancashire Hospitals NHS Trust,Acute - Large,3,RXR,C_999,57371,2368,8695,14741,58192.0,0.611685,2025,7,2025-q3


In [174]:

# average by provider/year/quarter
q = (
    df.groupby(["provider", "quarter"])["wait_pct_lt_18"]
    .mean()
    .reset_index()
)
q = q[q['quarter'].isin([f"{year}-q{quarter}" for year, quarter in full_quarters])]
q

# q['quarter'].isin([f"{year}-q{quarter}" for year, quarter in full_quarters])]

,provider,quarter,wait_pct_lt_18
14,R0B,2024-q4,0.746413
15,R0B,2025-q1,0.751532
16,R0B,2025-q2,0.754843
32,RAJ,2024-q4,0.521804
33,RAJ,2025-q1,0.502486
...,...,...,...
392,RXK,2025-q1,0.537904
393,RXK,2025-q2,0.565630
409,RXR,2024-q4,0.570337
410,RXR,2025-q1,0.573669


In [175]:

# pivot to quarters as columns
pivot = q.pivot(index="provider", columns=["quarter"], values="wait_pct_lt_18")
pivot



quarter,2024-q4,2025-q1,2025-q2
provider,,,
R0B,0.746413,0.751532,0.754843
RAJ,0.521804,0.502486,0.492790
RDE,0.549499,0.551230,0.560531
RDU,0.510425,0.516470,0.550233
REF,0.686029,0.688836,0.711441
RGN,0.524479,0.527045,0.534518
RH8,0.580356,0.599191,0.603163
RHU,0.542113,0.534613,0.554693
RHW,0.814172,0.779011,0.786013


In [176]:

# pivot.columns = [f"2024-Q{c}" for c in pivot.columns]

# compute trend vs 3 quarters earlier (or 2 here)
pivot["trend_delta"] = pivot.iloc[:, 0] - pivot.iloc[:, 2]
pivot


quarter,2024-q4,2025-q1,2025-q2,trend_delta
provider,,,,
R0B,0.746413,0.751532,0.754843,-0.008430
RAJ,0.521804,0.502486,0.492790,0.029014
RDE,0.549499,0.551230,0.560531,-0.011032
RDU,0.510425,0.516470,0.550233,-0.039808
REF,0.686029,0.688836,0.711441,-0.025413
RGN,0.524479,0.527045,0.534518,-0.010039
RH8,0.580356,0.599191,0.603163,-0.022807
RHU,0.542113,0.534613,0.554693,-0.012580
RHW,0.814172,0.779011,0.786013,0.028159


In [ ]:

pivot["Trend"] = pd.cut(
    pivot["trend_delta"],
    bins=[-1, -0.01, 0.01, 1],
    labels=["↓ Declining", "→ Stable", "↑ Improving"]
)

# format as percent
for col in range(3):
    pivot[f"col_{col}"] = (pivot.iloc[:, col] * 100).round(1).astype(str) + "%"

pivot = pivot.reset_index()

In [178]:

wide_df = pd.merge(pivot, providers, on='provider', how='inner')

print(wide_df)


   provider   2024-q4   2025-q1   2025-q2  trend_delta        Trend  col_0  \
0       R0B  0.746413  0.751532  0.754843    -0.008430     → Stable  74.6%   
1       RAJ  0.521804  0.502486  0.492790     0.029014  ↑ Improving  52.2%   
2       RDE  0.549499  0.551230  0.560531    -0.011032  ↓ Declining  54.9%   
3       RDU  0.510425  0.516470  0.550233    -0.039808  ↓ Declining  51.0%   
4       REF  0.686029  0.688836  0.711441    -0.025413  ↓ Declining  68.6%   
5       RGN  0.524479  0.527045  0.534518    -0.010039  ↓ Declining  52.4%   
6       RH8  0.580356  0.599191  0.603163    -0.022807  ↓ Declining  58.0%   
7       RHU  0.542113  0.534613  0.554693    -0.012580  ↓ Declining  54.2%   
8       RHW  0.814172  0.779011  0.786013     0.028159  ↑ Improving  81.4%   
9       RJ2  0.565563  0.570504  0.563829     0.001734     → Stable  56.6%   
10      RL4  0.527982  0.512908  0.539640    -0.011658  ↓ Declining  52.8%   
11      RN5  0.579802  0.585743  0.591612    -0.011810  ↓ Declin